In [1]:
import gc
from pickle import FALSE
import pandas as pd
from openpyxl.styles import Font, Alignment
from openpyxl import Workbook,utils
from openpyxl.worksheet.table import Table, TableStyleInfo
from procyclingstats import Race, Rider, Stage
from openpyxl.drawing.image import Image
from procyclingstats import RaceClimbs

# Helper: escribe un DataFrame con encabezados (str) y opcionalmente como tabla
# Para evitar autofiltros, no crear la tabla (add_table=False)
def write_df_to_sheet(ws, df, start_row, start_col, table_name, style_name="TableStyleMedium9",
                      show_first_col=False, show_last_col=False, show_row_stripes=True, show_col_stripes=False,
                      add_table=True):
    if df is None or df.empty:
        return start_row
    col_widths = {}
    # Encabezados
    for i, col in enumerate(df.columns, start=start_col):
        cell_value = str(col)
        ws.cell(row=start_row, column=i, value=cell_value)
        col_widths[i] = max(col_widths.get(i, 0), len(cell_value))
    # Datos
    for r_i, row in enumerate(df.itertuples(index=False, name=None), start=start_row + 1):
        for c_i, value in enumerate(row, start=start_col):
            ws.cell(row=r_i, column=c_i, value=value)
            value_len = len(str(value)) if value is not None else 0
            col_widths[c_i] = max(col_widths.get(c_i, 0), value_len)
    end_row = start_row + len(df)
    end_col = start_col + len(df.columns) - 1
    # Ajuste de ancho
    for col_idx, width in col_widths.items():
        col_letter = utils.get_column_letter(col_idx)
        current_width = ws.column_dimensions[col_letter].width
        ws.column_dimensions[col_letter].width = max(width + 2, current_width or 0)
    if add_table:
        ref = f"{utils.get_column_letter(start_col)}{start_row}:{utils.get_column_letter(end_col)}{end_row}"
        table = Table(displayName=table_name, ref=ref)
        table.tableStyleInfo = TableStyleInfo(
            name=style_name,
            showFirstColumn=show_first_col,
            showLastColumn=show_last_col,
            showRowStripes=show_row_stripes,
            showColumnStripes=show_col_stripes
        )
        ws.add_table(table)
    return end_row

def write_df_to_sheet_especialidad(ws, df, start_row, start_col):
    """Escribe cada fila del DataFrame de especialidades en una sola celda"""
    if df is None or df.empty:
        return start_row
    
    current_row = start_row
    max_width = 0
    
    # Procesar cada fila del DataFrame
    for idx, row in df.iterrows():
        # Concatenar las especialidades en una sola cadena
        cell_value = f"{row['specialty_1']}:{row['points_1']}, {row['specialty_2']}:{row['points_2']}"
        ws.cell(row=current_row, column=start_col, value=cell_value)
        max_width = max(max_width, len(cell_value))
        current_row += 1
    
    # Ajustar ancho de columna
    col_letter = utils.get_column_letter(start_col)
    ws.column_dimensions[col_letter].width = max(max_width + 2, ws.column_dimensions[col_letter].width or 0)
    
    return current_row

#enlace_o_valor="race/trofeo-calvia/2026" 
enlace_o_valor="race/ruta-de-la-ceramica-gran-premio-castellon/2026"
#carreras de un solo día  
race= Race(f"{enlace_o_valor}/overview") 
file=f"analisis_carrera_{race.name()}.xlsx"

libro=Workbook()
hojas=libro.active  

df_res = pd.DataFrame()  
df_fin = pd.DataFrame()
df_rider_specialidad = pd.DataFrame()
list_editions = race.prev_editions_select() 
c=0

img = Image('logo.png')
hojas.add_image(img, 'A1')
hojas.merge_cells('A1:C6')
img.height = 170
img.width = 410

hojas.title=race.name() # type: ignore
# Fusionar A1:E1 y aplicar fuente negra tamaño 16
hojas['D4'] = race.name()+" "+race.uci_tour()# type: ignore
hojas.merge_cells('D4:G4')
hojas['D4'].font = Font(color="F40BE4", size=30, bold=True) 
hojas['D4'].alignment = Alignment(horizontal="center", vertical="center")

hojas[f'A{hojas.max_row+2}']=f"EDICION:{race.year() }, Fecha:{race.startdate()}"
hojas[f'A{hojas.max_row}'].font = Font(color="000000", size=25, bold=True)
#obtenemos informacion de los puertos ediciaon actual
race_climbs = RaceClimbs(f"{enlace_o_valor}/route/climbs")
if race_climbs is not None:
    df_climbs_current = pd.DataFrame(race_climbs.climbs())
    #hojas[f'A{hojas.max_row}']=f"Puertos edición"
    end_row_res = write_df_to_sheet(
        hojas, df_climbs_current, start_row=hojas.max_row+1, start_col=1,
        table_name=f"Climbs_{race.name().replace('-', '_')}",
        style_name="TableStyleMedium5",
        add_table=True
    )
if(len(list_editions)>5):
    cont=6
else:
    cont=len(list_editions)
for edition in list_editions[1:cont]:
   
    print("Procesando edición", edition['text'])
    hojas[f'A{hojas.max_row+2}']=f"Edición {edition['text']}"
    hojas[f'A{hojas.max_row}'].font = Font(color="000000", size=16, bold=True)
    
  
    past_edit=Stage(f"{enlace_o_valor.strip()[0:len(enlace_o_valor)-4]}{edition['text']}/result")
    race_climbs = RaceClimbs(f"{enlace_o_valor.strip()[0:len(enlace_o_valor)-4]}{edition['text']}/route/climbs")
    df_climbs = pd.DataFrame(race_climbs.climbs()) if race_climbs is not None else pd.DataFrame()
    if not df_climbs.empty:
       df_climbs.rename(columns={'climb_name': 'Nombre', 'length': 'kms', 'steepness': 'Media','top':'altitud','km_before_finnish':'Distancia a Meta'}, inplace=True)
       hojas[f'A{hojas.max_row+1}']=f"Puertos edición"
       end_row_res=write_df_to_sheet(
         hojas, df_climbs[['Nombre', 'kms','Media','altitud','Distancia a Meta']].sort_values(by='Distancia a Meta', ascending=False), start_row=hojas.max_row+1, start_col=1,
         table_name=f"climbs_{edition['text'].replace('-', '_')}",
         style_name="TableStyleMedium2",
         add_table=True
       )
    print(f'como:{hojas.max_row},participacion:{past_edit.race_startlist_quality_score()} ,desnivel: {past_edit.vertical_meters()}m ,distancia {past_edit.distance()}km, avg:{past_edit.avg_speed_winner()}km/h')
    c=hojas.max_row+2
    hojas[f'A{c-1}']="Resumen:"
    hojas[f'A{c+1}']=f'{past_edit.won_how()}'
    hojas[f'A{c+1}'].font = Font(color="FF0000", bold=True)
    hojas[f'B{c+1}']=f'Participacion:{past_edit.race_startlist_quality_score()}'
    hojas[f'C{c+1}']=f'Desnivel: {past_edit.vertical_meters()}m'# type: ignore
    hojas[f'D{c+1}']=f'{past_edit.distance()}km'# type: ignore
    hojas[f'E{c+1}']=f'{past_edit.avg_speed_winner()}km/h'# type: ignore
    
    df_res = pd.DataFrame(past_edit.parse()['results'])# type: ignore
    
    if(df_res.empty):
        print("  No hay resultados para esta edición.")
        hojas.append(["  No hay resultados para esta edición."])# type: ignore
        continue
    
    # Obtener especialidad de los 10 primeros ciclistas
    df_riders_specialties = []
    for idx, row in df_res.head(10).iterrows():
        try:
            rider = Rider(str(row['rider_url']))
            rider_data = rider.parse()
            points_per_speciality = rider_data.get('points_per_speciality', {})
            
            if isinstance(points_per_speciality, dict) and points_per_speciality:
                sorted_by_values = dict(sorted(points_per_speciality.items(), key=lambda item: item[1], reverse=True))
                # Tomar las 2 primeras especialidades
                top_2 = list(sorted_by_values.items())[:2]
                rider_info = {                    
                    'specialty_1': top_2[0][0] if len(top_2) > 0 else '',
                    'points_1': top_2[0][1] if len(top_2) > 0 else 0,
                    'specialty_2': top_2[1][0] if len(top_2) > 1 else '',
                    'points_2': top_2[1][1] if len(top_2) > 1 else 0
                }
                df_riders_specialties.append(rider_info)
        except Exception as e:
            print(f"Error al procesar ciclista {row.get('rider_name', 'unknown')}: {e}")
    
    df_rider_specialidad = pd.DataFrame(df_riders_specialties)
    
    # Preparar tabla de resultados con especialidades
    df_res_subset = df_res[['rank', 'rider_name','time','team_name']].head(10).reset_index(drop=True)
    specialty_series = pd.Series([""] * len(df_res_subset))
    if not df_rider_specialidad.empty:
        specialty_series = df_rider_specialidad.apply(
            lambda r: f"{r['specialty_1']}:{r['points_1']}, {r['specialty_2']}:{r['points_2']}",
            axis=1
        ).reindex(df_res_subset.index).fillna("")
    df_res_subset['specialties'] = specialty_series
    
    #sacamos ciclistas Burgos y puntos UCI 
    df_burgos = df_res[df_res['team_name'].str.contains('Burgos')]
    df_uci=df_res.groupby('team_name')['uci_points'].sum().reset_index().sort_values(by='uci_points', ascending=False)
    
    # Escribir df_res con especialidades como última columna
    hojas[f'A{hojas.max_row+2}']="TOP 10"
    end_row_res = write_df_to_sheet(
        hojas, df_res_subset, start_row=hojas.max_row+2, start_col=1,
        table_name=f"Resultados_{edition['text'].replace('-', '_')}",
        style_name="TableStyleMedium5",
        add_table=True
    )
    
    # Escribir df_burgos debajo de df_res (A-E) sin tabla
    end_row_burgos = end_row_res
    if not df_burgos.empty:
        df_burgos_to_write = df_burgos[['rank', 'rider_name','time','uci_points','breakaway_kms']]
        hojas[f'A{hojas.max_row+2}']="RESULTADO BURGOS"
        end_row_burgos = write_df_to_sheet(
            hojas, df_burgos_to_write, start_row=end_row_res+3, start_col=1,
            table_name=f"Burgos_{edition['text'].replace('-', '_')}",
            style_name="TableStyleMedium5",
            show_first_col=True, show_col_stripes=True,
            add_table=True
        )
    
    # Escribir df_uci (G-H) sin tabla
    df_uci_to_write = df_uci
    hojas[f'G{c+2}']="PUNTOS UCI POR EQUIPO"
    end_row_uci = write_df_to_sheet(
        hojas, df_uci_to_write, start_row=c+3, start_col=7,
        table_name=f"UCI_{edition['text'].replace('-', '_')}",
        style_name="TableStyleMedium5",
        show_first_col=True,
        add_table=True
    )
    
    # Avanzar c
    #c = max(end_row_burgos, end_row_specialidad, end_row_uci) + 2
    gc.collect() 
  
libro.save (file)

c:\Users\echav\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


Procesando edición 2025
como:15,participacion:(246, 246) ,desnivel: 2374m ,distancia 171.7km, avg:43.063km/h
#####################25 January 2025
Error al procesar ciclista Dujardin Sandy: list index out of range
Procesando edición 2024


WebDriverException: Message: target frame detached
  (failed to check if window was closed: disconnected: Unable to receive message from renderer)
  (Session info: chrome=143.0.7499.169)
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff64a5388e5
	0x7ff64a538940
	0x7ff64a311465
	0x7ff64a2fe550
	0x7ff64a2fd40a
	0x7ff64a3227af
	0x7ff64a321a88
	0x7ff64a3bab71
	0x7ff64a35ac29
	0x7ff64a35ba93
	0x7ff64a850640
	0x7ff64a84af80
	0x7ff64a8696e6
	0x7ff64a555de4
	0x7ff64a55ed8c
	0x7ff64a542004
	0x7ff64a5421b5
	0x7ff64a527ee2
	0x7ffbe8ee5c07
	0x7ffbe9dd9c70


In [ ]:
#carreras por etapas
enlace_o_valor="race/itzulia-basque-country/2026"
import gc
 

print("Intentando obtener resultados para:", enlace_o_valor)
race= Race(f"{enlace_o_valor}/overview")     
df_res = pd.DataFrame()  
df_fin = pd.DataFrame()
list_editions = race.prev_editions_select() 
etapas=race.stages()
print(list_editions)

for edition in list_editions[1:5]:
   
    race= Race(f"{edition['value']}/overview")  
    print("========================================")
    print("Procesando edición", edition['text'])
    print("========================================")
    etapas=race.stages()
    for etapa in etapas:
        print(f"{etapa['stage_url']}/result")
        past_edit=Stage(f"{etapa['stage_url']}/result")
        #print(past_edit.parse().keys())
        print(past_edit.departure()+" - "+past_edit.arrival())
        print(f'como:{past_edit.won_how()} ,participacion:{past_edit.race_startlist_quality_score()},desnivel: {past_edit.vertical_meters()}m ,distancia {past_edit.distance()}km, avg:{past_edit.avg_speed_winner()}km/h')
        df_res = pd.DataFrame(past_edit.parse()['results'])
    
        #print(df_res['rider_url'].head(1))
        rider=Rider(str(df_res['rider_url'].head(1).values[0])) # type: ignore
    
        points_per_speciality = rider.parse()['points_per_speciality']
        if isinstance(points_per_speciality, dict):
            sorted_by_values = dict(sorted(points_per_speciality.items(), key=lambda item: item[1], reverse=True))
            df_rider_specialidad = pd.DataFrame([sorted_by_values])
        else:
            df_rider_specialidad = pd.DataFrame(points_per_speciality)
   
        print(df_rider_specialidad.keys()[0]+":"+str(df_rider_specialidad.iloc[0,0])+","+
        df_rider_specialidad.keys()[1]+":"+str(df_rider_specialidad.iloc[0,1]))
          
        print(df_res[['rank', 'rider_name','time','team_name']].head(5))
        df=df_res.groupby('team_name')['uci_points'].sum().reset_index().sort_values(by='uci_points', ascending=False)
        #print(df)
        df_burgos = df_res[df_res['team_name'].str.contains('Burgos')]
     
        print(df_burgos[['rank', 'rider_name', 'time', 'breakaway_kms','uci_points']].head(4))   
        gc.collect() 

       

Intentando obtener resultados para: race/itzulia-basque-country/2026
[{'text': '2026', 'value': 'race/itzulia-basque-country/2026/statistics/start'}, {'text': '2025', 'value': 'race/itzulia-basque-country/2025/statistics/start'}, {'text': '2024', 'value': 'race/itzulia-basque-country/2024/statistics/start'}, {'text': '2023', 'value': 'race/itzulia-basque-country/2023/statistics/start'}, {'text': '2022', 'value': 'race/itzulia-basque-country/2022/statistics/start'}, {'text': '2021', 'value': 'race/itzulia-basque-country/2021/statistics/start'}, {'text': '2020', 'value': 'race/itzulia-basque-country/2020/statistics/start'}, {'text': '2019', 'value': 'race/itzulia-basque-country/2019/statistics/start'}, {'text': '2018', 'value': 'race/itzulia-basque-country/2018/statistics/start'}, {'text': '2017', 'value': 'race/itzulia-basque-country/2017/statistics/start'}, {'text': '2016', 'value': 'race/itzulia-basque-country/2016/statistics/start'}, {'text': '2015', 'value': 'race/itzulia-basque-cou

KeyboardInterrupt: 